In [1]:
import random

# 1. Define the Grammar in CNF
# Rules: A -> (B, C) or A -> terminal_string
grammar = {
    "S":   [("NP", "VP")],
    "VP":  [("V", "NP")],
    "NP":  [("Det", "N"), ("Adj", "N")],
    "Det": ["the", "a"],
    "N":   ["cat", "dog", "fish"],
    "V":   ["eats", "sees", "likes"],
    "Adj": ["big", "small"]
}

# 2. Random Sampling (Sentence Generation)
def generate(symbol):
    if symbol not in grammar:
        return symbol
    expansion = random.choice(grammar[symbol])
    if isinstance(expansion, tuple):
        return f"{generate(expansion[0])} {generate(expansion[1])}"
    return expansion

# 3. CYK Algorithm with Backpointers
def cyk_parse(sentence):
    words = sentence.split()
    n = len(words)
    # table[i][j] stores sets of (NonTerminal, LeftChild, RightChild, SplitPoint)
    table = [[[] for _ in range(n + 1)] for _ in range(n + 1)]

    # Fill diagonal (Terminals)
    for i, word in enumerate(words):
        for lhs, rhs in grammar.items():
            if word in rhs:
                table[i][i+1].append((lhs, word, None, None))

    # Fill table (Non-terminals)
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length
            for k in range(i + 1, j):
                for lhs, rhs in grammar.items():
                    for prod in rhs:
                        if isinstance(prod, tuple):
                            B_rules = [x for x in table[i][k] if x[0] == prod[0]]
                            C_rules = [x for x in table[k][j] if x[0] == prod[1]]
                            for b in B_rules:
                                for c in C_rules:
                                    table[i][j].append((lhs, b, c, k))
    return table, words

# 4. Tree Visualization (Recursive)
def print_tree(node, indent=0):
    symbol, left, right, _ = node
    print("  " * indent + f"({symbol}", end="")
    if right is None: # Terminal leaf
        print(f": {left})")
    else:
        print("")
        print_tree(left, indent + 1)
        print_tree(right, indent + 1)
        print("  " * indent + ")")

# Execution
sentence = generate("S")
print(f"Generated Sentence: {sentence}\n")

table, words = cyk_parse(sentence)
final_parses = [node for node in table[0][len(words)] if node[0] == "S"]

if final_parses:
    print("Parse Tree Structure:")
    print_tree(final_parses[0])
else:
    print("No valid parse found.")

Generated Sentence: the cat eats small fish

Parse Tree Structure:
(S
  (NP
    (Det: the)
    (N: cat)
  )
  (VP
    (V: eats)
    (NP
      (Adj: small)
      (N: fish)
    )
  )
)
